# Original BoL Feature Tabular Baseline

This notebook uses the **original BoL approach** feature index, not the former fixed pre-2000 temporal baseline.

Original BoL feature index:

- 864 year-specific predictor variables
- survey years from 1986 to 2020
- features grouped into behavioral/externalizing, prior delinquency/contact, substance/risk behavior, school/achievement, family/home, etc.

Because the original BoL approach contains features from the same broad period as the target window, we mirror the original BoL cutoff logic instead of using all columns blindly. For every person, predictors are kept only if they occur before that person's cutoff year.

Cutoff logic:

- positive cases: cutoff is the second positive delinquency/contact event year
- negative cases: cutoff is the last observed target year

This avoids using information at or after the event point inside the tabular feature matrix.

## Model: Gradient Boosting with Decision Stumps

This model is a lightweight nonlinear baseline using the original BoL feature index after person-specific cutoff censoring.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def find_project_dir(start=None):
    """Find the project root independent of where Jupyter was launched."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        has_raw_data = (candidate / "nlsy79_child_youngadult_selected_crime_features.csv").exists()
        has_tabular_dir = (candidate / "Tabular_based_models").exists()
        has_bol_dir = (candidate / "BoL approach").exists()
        if has_raw_data and has_tabular_dir and has_bol_dir:
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Start Jupyter inside the Bol_Crime project folder "
        "or one of its subfolders."
    )


PROJECT_DIR = find_project_dir()
TABULAR_DIR = PROJECT_DIR / "Tabular_based_models"
MODEL_DIR = TABULAR_DIR / "gradient_boosting"

DATA_PATH = PROJECT_DIR / "nlsy79_child_youngadult_selected_crime_features.csv"
FEATURE_INDEX_PATH = PROJECT_DIR / "BoL approach" / "metadata_examples" / "child_crime_broad_persistent_feature_index.csv"
TARGETS_PATH = TABULAR_DIR / "data" / "targets" / "nlsy79_temporal_delinquency_targets_2000_2020.csv"
OUT_DIR = MODEL_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_PATHS = {
    "raw data": DATA_PATH,
    "original BoL feature index": FEATURE_INDEX_PATH,
    "target csv": TARGETS_PATH,
}
missing_paths = {name: path for name, path in REQUIRED_PATHS.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(f"{name}: {path}" for name, path in missing_paths.items()))

TARGET = "later_persistent_delinquency_contact_2000_2020"
SECOND_EVENT_YEAR = "later_delinquency_contact_second_event_year_2000_2020"
LAST_OBSERVED_YEAR = "later_delinquency_contact_last_observed_year_2000_2020"
MISSING_CODES = {-1, -2, -3, -4, -5, -7}
RANDOM_SEED = 2026
TEST_SIZE = 0.30
FEATURE_SOURCE = "original BoL broad persistent feature index"
CUTOFF_RULE = "positive: before second event year; negative: before last observed target year"

print("Project dir:", PROJECT_DIR)
print("Target file:", TARGETS_PATH)


## Load Original BoL Features and Target

We use the original 864-feature BoL index and the temporally defined persistent delinquency/contact target.

In [ ]:
data = pd.read_csv(DATA_PATH)
feature_index = pd.read_csv(FEATURE_INDEX_PATH)
targets = pd.read_csv(TARGETS_PATH)

feature_cols = [c for c in feature_index["csv_code"].tolist() if c in data.columns]
feature_year = feature_index.set_index("csv_code")["survey_year"].to_dict()

model_df = data[["C0000100"] + feature_cols].merge(
    targets[["C0000100", TARGET, SECOND_EVENT_YEAR, LAST_OBSERVED_YEAR]],
    on="C0000100",
    how="inner",
)
model_df = model_df[model_df[TARGET].notna()].copy()
model_df["cutoff_year"] = np.where(
    model_df[TARGET].eq(1),
    model_df[SECOND_EVENT_YEAR],
    model_df[LAST_OBSERVED_YEAR],
)
model_df = model_df[model_df["cutoff_year"].notna()].copy()

print("Rows:", len(model_df))
print("Original BoL features available:", len(feature_cols))
print("Target balance:")
print(model_df[TARGET].value_counts().sort_index())
print("Base rate:", round(model_df[TARGET].mean(), 3))
print("Cutoff rule:", CUTOFF_RULE)

feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]].head(10)


## Preprocessing

Negative NLSY missing codes are recoded to NA. Features at or after each person's cutoff year are also set to NA. Missing values are median-imputed using the training set only. Gradient boosting uses the imputed values directly, without standardization.

In [ ]:
def clean_feature_matrix_with_cutoff(df, feature_cols, feature_year):
    x = df[feature_cols].copy()
    cutoff = pd.to_numeric(df["cutoff_year"], errors="coerce")

    for col in feature_cols:
        x[col] = pd.to_numeric(x[col], errors="coerce")
        x[col] = x[col].replace([np.inf, -np.inf], np.nan)
        x.loc[x[col].isin(MISSING_CODES), col] = np.nan

        year = feature_year.get(col)
        if str(year) != "XRND":
            year_num = pd.to_numeric(pd.Series([year]), errors="coerce").iloc[0]
            if pd.notna(year_num):
                # Mirror original BoL: only information before the person-specific cutoff is allowed.
                x.loc[cutoff <= year_num, col] = np.nan
    return x


def train_test_split_stratified(y, test_size=0.30, seed=2026):
    rng = np.random.default_rng(seed)
    train_idx = []
    test_idx = []
    for label in sorted(np.unique(y)):
        idx = np.where(y == label)[0]
        rng.shuffle(idx)
        n_test = int(round(len(idx) * test_size))
        test_idx.extend(idx[:n_test].tolist())
        train_idx.extend(idx[n_test:].tolist())
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return np.array(train_idx), np.array(test_idx)


def fit_median_imputer(x_train):
    return x_train.median(axis=0, skipna=True).fillna(0.0)


def impute_with_median(x, medians):
    arr = x.fillna(medians).to_numpy(dtype=float)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def fit_standardizer(x):
    means = x.mean(axis=0)
    stds = x.std(axis=0)
    stds[stds == 0] = 1.0
    return means, stds


def standardize(x, means, stds):
    z = (x - means) / stds
    return np.clip(np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0), -10.0, 10.0)


def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def auc_score(y_true, prob):
    order = np.argsort(prob)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(prob) + 1)
    pos = y_true == 1
    n_pos = int(pos.sum())
    n_neg = int((~pos).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    rank_sum_pos = ranks[pos].sum()
    return float((rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    return {
        "threshold": threshold,
        "accuracy": float((pred == y_true).mean()),
        "auc": auc_score(y_true, prob),
        "sensitivity_tpr": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity_tnr": tn / (tn + fp) if (tn + fp) else np.nan,
        "predicted_positive_rate": float(pred.mean()),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


In [ ]:
y = model_df[TARGET].astype(int).to_numpy()
x_raw = clean_feature_matrix_with_cutoff(model_df, feature_cols, feature_year)

train_idx, test_idx = train_test_split_stratified(y, TEST_SIZE, RANDOM_SEED)

x_train_raw = x_raw.iloc[train_idx]
x_test_raw = x_raw.iloc[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

medians = fit_median_imputer(x_train_raw)
x_train_imputed = impute_with_median(x_train_raw, medians)
x_test_imputed = impute_with_median(x_test_raw, medians)

means, stds = fit_standardizer(x_train_imputed)
x_train_std = standardize(x_train_imputed, means, stds)
x_test_std = standardize(x_test_imputed, means, stds)

print("Train N:", len(y_train), "Test N:", len(y_test))
print("Train base rate:", round(y_train.mean(), 3))
print("Test base rate:", round(y_test.mean(), 3))
print("Feature matrix shape:", x_train_imputed.shape)


## Fit Gradient Boosting with Decision Stumps

In [ ]:
N_ESTIMATORS = 150
GB_LEARNING_RATE = 0.05
MAX_THRESHOLDS_PER_FEATURE = 24
MIN_LEAF_SIZE = 30


def logit(p):
    p = min(max(float(p), 1e-6), 1 - 1e-6)
    return float(np.log(p / (1 - p)))


def candidate_thresholds(values):
    unique = np.unique(values)
    if len(unique) <= 1:
        return np.array([])
    if len(unique) <= MAX_THRESHOLDS_PER_FEATURE:
        return (unique[:-1] + unique[1:]) / 2
    qs = np.linspace(0.02, 0.98, MAX_THRESHOLDS_PER_FEATURE)
    return np.unique(np.quantile(values, qs))


def fit_best_stump(x, residual, feature_names):
    best = {
        "feature_index": None,
        "feature": None,
        "threshold": None,
        "left_value": 0.0,
        "right_value": 0.0,
        "loss": np.inf,
    }
    n = len(residual)
    for j in range(x.shape[1]):
        values = x[:, j]
        for threshold in candidate_thresholds(values):
            left = values <= threshold
            n_left = int(left.sum())
            n_right = n - n_left
            if n_left < MIN_LEAF_SIZE or n_right < MIN_LEAF_SIZE:
                continue
            left_value = float(residual[left].mean())
            right_value = float(residual[~left].mean())
            pred = np.where(left, left_value, right_value)
            loss = float(np.mean((residual - pred) ** 2))
            if loss < best["loss"]:
                best = {
                    "feature_index": j,
                    "feature": feature_names[j],
                    "threshold": float(threshold),
                    "left_value": left_value,
                    "right_value": right_value,
                    "loss": loss,
                }
    if best["feature_index"] is None:
        raise RuntimeError("Could not find a valid stump split.")
    return best


def predict_stump(x, stump):
    values = x[:, int(stump["feature_index"])]
    return np.where(values <= stump["threshold"], stump["left_value"], stump["right_value"])


def fit_gradient_boosting(x, y, feature_names):
    base_score = logit(y.mean())
    f = np.full(len(y), base_score, dtype=float)
    stumps = []
    history = []
    for iteration in range(N_ESTIMATORS):
        p = sigmoid(f)
        residual = y - p
        stump = fit_best_stump(x, residual, feature_names)
        f += GB_LEARNING_RATE * predict_stump(x, stump)
        stump["iteration"] = iteration + 1
        stumps.append(stump)

        eps = 1e-12
        p = sigmoid(f)
        loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        history.append(float(loss))
    return base_score, stumps, history


def predict_gradient_boosting(x, base_score, stumps):
    f = np.full(len(x), base_score, dtype=float)
    for stump in stumps:
        f += GB_LEARNING_RATE * predict_stump(x, stump)
    return sigmoid(f)

gb_base_score, gb_stumps, gb_history = fit_gradient_boosting(x_train_imputed, y_train, feature_cols)
gb_train_prob = predict_gradient_boosting(x_train_imputed, gb_base_score, gb_stumps)
gb_test_prob = predict_gradient_boosting(x_test_imputed, gb_base_score, gb_stumps)

gb_train_metrics = metrics(y_train, gb_train_prob)
gb_test_metrics = metrics(y_test, gb_test_prob)

print("Estimators:", len(gb_stumps))
print("Final train loss:", round(gb_history[-1], 4))
print("Test metrics:")
print(pd.Series(gb_test_metrics).to_string())


## Save Outputs

In [ ]:
test_ids = model_df.iloc[test_idx]["C0000100"].astype(int).to_numpy()

pred_df = pd.DataFrame({
    "C0000100": test_ids,
    "target": TARGET,
    "model": "gradient_boosting_stumps_original_bol_features_cutoff",
    "y_true": y_test,
    "probability": gb_test_prob,
    "prediction": (gb_test_prob >= 0.5).astype(int),
})
pred_df["correct"] = pred_df["prediction"] == pred_df["y_true"]

stumps_df = pd.DataFrame(gb_stumps).merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    left_on="feature",
    right_on="csv_code",
    how="left",
)

importance_df = (
    stumps_df
    .groupby(["feature", "ref_id", "variable", "survey_year", "feature_group", "question"], dropna=False)
    .agg(n_splits=("iteration", "count"), mean_split_loss=("loss", "mean"))
    .reset_index()
    .sort_values(["n_splits", "mean_split_loss"], ascending=[False, True])
)
# Alias used by the plotting section for readability/reproducibility.
split_importance_df = importance_df

metrics_df = pd.DataFrame([
    {"split": "train", **gb_train_metrics},
    {"split": "test", **gb_test_metrics},
])

summary = {
    "target": TARGET,
    "model": "gradient_boosting_stumps_original_bol_features_cutoff",
    "feature_source": FEATURE_SOURCE,
    "cutoff_rule": CUTOFF_RULE,
    "n_total": int(len(model_df)),
    "n_train": int(len(train_idx)),
    "n_test": int(len(test_idx)),
    "n_features": int(len(feature_cols)),
    "test_size": TEST_SIZE,
    "random_seed": RANDOM_SEED,
    "n_estimators": N_ESTIMATORS,
    "learning_rate": GB_LEARNING_RATE,
    "max_thresholds_per_feature": MAX_THRESHOLDS_PER_FEATURE,
    "min_leaf_size": MIN_LEAF_SIZE,
    "initial_score": gb_base_score,
    "final_train_loss": gb_history[-1],
    "train_metrics": gb_train_metrics,
    "test_metrics": gb_test_metrics,
}

pred_df.to_csv(OUT_DIR / "gradient_boosting_stumps_predictions.csv", index=False)
stumps_df.to_csv(OUT_DIR / "gradient_boosting_stumps_splits.csv", index=False)
importance_df.to_csv(OUT_DIR / "gradient_boosting_stumps_feature_importance.csv", index=False)
metrics_df.to_csv(OUT_DIR / "gradient_boosting_stumps_metrics.csv", index=False)
(OUT_DIR / "gradient_boosting_stumps_summary.json").write_text(json.dumps(summary, indent=2))

importance_df.head(15)


## Threshold Evaluation and Fairness by Sex/Race

This section mirrors the BoL approach threshold and fairness analysis for the tabular model. The primary comparison threshold remains `0.50` because it is fixed before looking at subgroup outcomes. The best-accuracy threshold is reported as exploratory because it is selected on the same test predictions.


In [ ]:
import matplotlib.pyplot as plt

try:
    from IPython.display import display as safe_display
except Exception:
    def safe_display(obj):
        print(obj.to_string(index=False) if hasattr(obj, "to_string") else obj)

MODEL_LABEL = "Gradient Boosting Stumps"
OUTPUT_PREFIX = "gradient_boosting_stumps"
PRIMARY_THRESHOLD = 0.50

RACE_LABELS = {1: "Hispanic", 2: "Black", 3: "Non-Black/non-Hispanic"}
SEX_LABELS = {1: "Male", 2: "Female"}


def metrics_extended(y_true, prob, threshold=0.5):
    out = metrics(y_true, prob, threshold)
    tp, tn, fp, fn = out["tp"], out["tn"], out["fp"], out["fn"]
    out["fpr"] = fp / (fp + tn) if (fp + tn) else np.nan
    out["fnr"] = fn / (fn + tp) if (fn + tp) else np.nan
    out["ppv"] = tp / (tp + fp) if (tp + fp) else np.nan
    out["npv"] = tn / (tn + fn) if (tn + fn) else np.nan
    return out


def threshold_sweep(predictions):
    y = predictions["y_true"].astype(int).to_numpy()
    p = predictions["probability"].astype(float).to_numpy()
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    exact = np.round(np.unique(p), 6)
    thresholds = np.unique(np.concatenate([grid, exact]))
    rows = [metrics_extended(y, p, float(t)) for t in thresholds]
    out = pd.DataFrame(rows)
    return out.sort_values(["accuracy", "sensitivity_tpr", "specificity_tnr"], ascending=False).reset_index(drop=True)


def add_demographics(predictions):
    demo = data[["C0000100", "C0005300", "C0005400"]].copy()
    demo["race"] = pd.to_numeric(demo["C0005300"], errors="coerce").map(RACE_LABELS)
    demo["sex"] = pd.to_numeric(demo["C0005400"], errors="coerce").map(SEX_LABELS)
    out = predictions.merge(demo[["C0000100", "race", "sex"]], on="C0000100", how="left")
    return out


def fairness_rows(predictions, threshold, threshold_name):
    rows = []
    for attr_col, attr_label in [("sex", "Sex"), ("race", "Race")]:
        for subgroup, sub in predictions.dropna(subset=[attr_col]).groupby(attr_col):
            y = sub["y_true"].astype(int).to_numpy()
            p = sub["probability"].astype(float).to_numpy()
            m = metrics_extended(y, p, threshold)
            rows.append({
                "model": MODEL_LABEL,
                "threshold_name": threshold_name,
                "threshold": threshold,
                "attribute": attr_label,
                "subgroup": subgroup,
                "n": int(len(sub)),
                "base_rate": float(np.mean(y)),
                **m,
            })
    return pd.DataFrame(rows)


def fairness_gap_table(fairness_df):
    metrics_to_compare = ["base_rate", "predicted_positive_rate", "accuracy", "auc", "sensitivity_tpr", "specificity_tnr", "fpr", "fnr", "ppv", "npv"]
    rows = []
    for (threshold_name, attribute), g in fairness_df.groupby(["threshold_name", "attribute"]):
        row = {"model": MODEL_LABEL, "threshold_name": threshold_name, "attribute": attribute}
        for metric_name in metrics_to_compare:
            vals = g[metric_name].dropna()
            row[f"{metric_name}_gap"] = float(vals.max() - vals.min()) if len(vals) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


threshold_eval_df = threshold_sweep(pred_df)
best_accuracy_threshold = float(threshold_eval_df.iloc[0]["threshold"])

pred_with_demo_df = add_demographics(pred_df)
fairness_primary_df = fairness_rows(pred_with_demo_df, PRIMARY_THRESHOLD, "fixed_0.50")
fairness_best_df = fairness_rows(pred_with_demo_df, best_accuracy_threshold, "test_best_accuracy_exploratory")
fairness_df = pd.concat([fairness_primary_df, fairness_best_df], ignore_index=True)
fairness_gaps_df = fairness_gap_table(fairness_df)

threshold_eval_df.to_csv(OUT_DIR / f"{OUTPUT_PREFIX}_threshold_eval.csv", index=False)
fairness_df.to_csv(OUT_DIR / f"{OUTPUT_PREFIX}_fairness_metrics.csv", index=False)
fairness_gaps_df.to_csv(OUT_DIR / f"{OUTPUT_PREFIX}_fairness_gaps.csv", index=False)

print("Best exploratory threshold by test accuracy:", round(best_accuracy_threshold, 3))
print("Primary fixed threshold for comparison:", PRIMARY_THRESHOLD)
safe_display(threshold_eval_df.head(10))
safe_display(fairness_df[fairness_df["threshold_name"].eq("fixed_0.50")])
safe_display(fairness_gaps_df)

# Plot threshold sweep: top thresholds by accuracy.
threshold_plot = threshold_eval_df.head(12).copy()
threshold_plot = threshold_plot.iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 7))
y_pos = np.arange(len(threshold_plot))
bar_height = 0.24
ax.barh(y_pos - bar_height, threshold_plot["accuracy"], height=bar_height, label="accuracy", color="#59a14f")
ax.barh(y_pos, threshold_plot["sensitivity_tpr"], height=bar_height, label="sensitivity_tpr", color="#72a17a")
ax.barh(y_pos + bar_height, threshold_plot["fpr"], height=bar_height, label="fpr", color="#a65f46")
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{t:.2f}" for t in threshold_plot["threshold"]])
ax.set_xlim(0, 1)
ax.set_xlabel("metric value")
ax.set_ylabel("threshold")
ax.set_title(f"Threshold sweep for {MODEL_LABEL}")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{OUTPUT_PREFIX}_threshold_sweep_top12.png", dpi=200, bbox_inches="tight")
plt.show()

# Plot fairness at fixed threshold 0.50.
fairness_plot = fairness_primary_df.copy()
fairness_plot["group"] = fairness_plot["attribute"] + ": " + fairness_plot["subgroup"].astype(str)
fairness_plot = fairness_plot.iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 7))
y_pos = np.arange(len(fairness_plot))
bar_height = 0.22
ax.barh(y_pos - bar_height, fairness_plot["accuracy"], height=bar_height, label="accuracy", color="#59a14f")
ax.barh(y_pos, fairness_plot["sensitivity_tpr"], height=bar_height, label="tpr", color="#72a17a")
ax.barh(y_pos + bar_height, fairness_plot["fpr"], height=bar_height, label="fpr", color="#a65f46")
ax.set_yticks(y_pos)
ax.set_yticklabels(fairness_plot["group"])
ax.set_xlim(0, 1)
ax.set_xlabel("metric value")
ax.set_title(f"Fairness metrics by subgroup for {MODEL_LABEL} at threshold 0.50")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{OUTPUT_PREFIX}_fairness_fixed_0_50.png", dpi=200, bbox_inches="tight")
plt.show()


## Feature Importance and Additive SHAP-style Attributions

This section adds three explainability outputs.

1. **Split-count importance:** already saved from the fitted boosting stumps.
2. **Permutation importance:** shuffle one feature in the test set and measure how much AUC/accuracy drops.
3. **Additive stump SHAP-style attributions:** the model is a sum of decision stumps, so each stump contribution can be assigned to the feature used by that stump. We center each stump around its average training contribution, then sum by feature. This gives a local and global additive explanation in logit space.

In [ ]:
def permutation_importance_gb(x_test, y_test, base_score, stumps, feature_cols, n_repeats=1, seed=2026):
    rng = np.random.default_rng(seed)
    baseline_prob = predict_gradient_boosting(x_test, base_score, stumps)
    baseline = metrics(y_test, baseline_prob)
    rows = []
    for j, feature in enumerate(feature_cols):
        aucs = []
        accs = []
        for _ in range(n_repeats):
            x_perm = x_test.copy()
            x_perm[:, j] = rng.permutation(x_perm[:, j])
            prob = predict_gradient_boosting(x_perm, base_score, stumps)
            m = metrics(y_test, prob)
            aucs.append(m["auc"])
            accs.append(m["accuracy"])
        rows.append({
            "csv_code": feature,
            "baseline_auc": baseline["auc"],
            "permuted_auc": float(np.mean(aucs)),
            "auc_drop": baseline["auc"] - float(np.mean(aucs)),
            "baseline_accuracy": baseline["accuracy"],
            "permuted_accuracy": float(np.mean(accs)),
            "accuracy_drop": baseline["accuracy"] - float(np.mean(accs)),
        })
    return pd.DataFrame(rows)

perm_importance_df = permutation_importance_gb(x_test_imputed, y_test, gb_base_score, gb_stumps, feature_cols, n_repeats=1, seed=RANDOM_SEED)
perm_importance_df = perm_importance_df.merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    on="csv_code",
    how="left",
).sort_values("auc_drop", ascending=False)

# Additive stump SHAP-style attributions in logit space.
feature_to_index = {feature: i for i, feature in enumerate(feature_cols)}
stump_train_means = []
for stump in gb_stumps:
    train_pred = predict_stump(x_train_imputed, stump)
    stump_train_means.append(float(train_pred.mean()))

base_logit = float(gb_base_score + GB_LEARNING_RATE * np.sum(stump_train_means))
raw_f = np.full(len(x_test_imputed), gb_base_score, dtype=float)
for stump in gb_stumps:
    raw_f += GB_LEARNING_RATE * predict_stump(x_test_imputed, stump)

stump_shap_values = np.zeros((len(x_test_imputed), len(feature_cols)), dtype=float)
for stump, train_mean in zip(gb_stumps, stump_train_means):
    j = int(stump["feature_index"])
    contribution = GB_LEARNING_RATE * (predict_stump(x_test_imputed, stump) - train_mean)
    stump_shap_values[:, j] += contribution

stump_shap_global_df = pd.DataFrame({
    "csv_code": feature_cols,
    "mean_abs_shap_logit": np.mean(np.abs(stump_shap_values), axis=0),
    "mean_shap_logit": np.mean(stump_shap_values, axis=0),
}).merge(
    importance_df[["feature", "n_splits", "mean_split_loss"]].rename(columns={"feature": "csv_code"}),
    on="csv_code",
    how="left",
).merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    on="csv_code",
    how="left",
)
stump_shap_global_df["n_splits"] = stump_shap_global_df["n_splits"].fillna(0).astype(int)
stump_shap_global_df = stump_shap_global_df.sort_values("mean_abs_shap_logit", ascending=False)
# Alias used by plotting cells for readability/reproducibility.
shap_global_df = stump_shap_global_df

local_rows = []
TOP_LOCAL_FEATURES = 8
for row_pos, child_id in enumerate(test_ids):
    top_idx = np.argsort(np.abs(stump_shap_values[row_pos]))[::-1][:TOP_LOCAL_FEATURES]
    for rank, j in enumerate(top_idx, start=1):
        local_rows.append({
            "C0000100": int(child_id),
            "rank": rank,
            "csv_code": feature_cols[j],
            "shap_logit": float(stump_shap_values[row_pos, j]),
            "abs_shap_logit": float(abs(stump_shap_values[row_pos, j])),
            "base_logit": base_logit,
            "predicted_logit": float(raw_f[row_pos]),
            "probability": float(gb_test_prob[row_pos]),
            "y_true": int(y_test[row_pos]),
        })
stump_shap_local_df = pd.DataFrame(local_rows).merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    on="csv_code",
    how="left",
)

perm_importance_df.to_csv(OUT_DIR / "gradient_boosting_stumps_permutation_importance.csv", index=False)
stump_shap_global_df.to_csv(OUT_DIR / "gradient_boosting_stumps_shap_style_global_importance.csv", index=False)
stump_shap_local_df.to_csv(OUT_DIR / "gradient_boosting_stumps_shap_style_local_top_contributions.csv", index=False)

print("Saved permutation importance and additive stump SHAP-style outputs.")
print("Top additive stump SHAP-style global features:")
stump_shap_global_df.head(15)


## Top 10 Feature Importance Plots

These plots visualize the ten most important features for the gradient boosting model using three views: split count, permutation AUC drop, and additive SHAP-style global importance.


In [ ]:
import matplotlib.pyplot as plt
import textwrap


def short_label(row, max_width=42):
    year = row.get("survey_year", "")
    var = row.get("variable", row.get("csv_code", row.get("feature", "")))
    question = str(row.get("question", ""))
    label = f"{var} ({year}) - {question}"
    return "\n".join(textwrap.wrap(label, width=max_width))


def plot_top10_barh(df, metric, title, filename):
    top = df.sort_values(metric, ascending=False).head(10).copy()
    top = top.iloc[::-1]
    labels = [short_label(row) for _, row in top.iterrows()]
    values = top[metric].to_numpy()

    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(range(len(top)), values, color="#59a14f")
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel(metric)
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


plot_top10_barh(
    importance_df,
    "n_splits",
    "Top 10 Gradient Boosting Features by Split Count",
    "gradient_boosting_top10_split_importance.png",
)

plot_top10_barh(
    perm_importance_df,
    "auc_drop",
    "Top 10 Gradient Boosting Features by Permutation Importance (AUC Drop)",
    "gradient_boosting_top10_permutation_importance.png",
)

plot_top10_barh(
    stump_shap_global_df,
    "mean_abs_shap_logit",
    "Top 10 Gradient Boosting Features by Additive SHAP-style Importance",
    "gradient_boosting_top10_shap_style_importance.png",
)


## SHAP-style XAI Summary Plots

These plots focus only on the additive stump SHAP-style explanations for gradient boosting. The bar plot shows global importance. The dot plot shows the distribution of SHAP-style logit contributions across test cases for the top features.


In [ ]:
# SHAP-style XAI plots for gradient boosting.
# These are additive stump SHAP-style values in logit space, computed above as stump_shap_values.

import matplotlib.pyplot as plt
import numpy as np
import textwrap


def shap_label(row, max_width=38):
    year = row.get("survey_year", "")
    var = row.get("variable", row.get("csv_code", ""))
    question = str(row.get("question", ""))
    return "\n".join(textwrap.wrap(f"{var} ({year}) - {question}", width=max_width))


def plot_shap_bar(global_df, metric, title, filename, color="#59a14f"):
    top = global_df.sort_values(metric, ascending=False).head(10).iloc[::-1].copy()
    labels = [shap_label(row) for _, row in top.iterrows()]
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(range(len(top)), top[metric], color=color)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("mean absolute SHAP-style contribution (logit)")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


def plot_shap_summary_dot(shap_values, feature_values, global_df, title, filename, top_n=10):
    top_codes = global_df.sort_values("mean_abs_shap_logit", ascending=False).head(top_n)["csv_code"].tolist()
    top_indices = [feature_cols.index(code) for code in top_codes]
    label_lookup = global_df.set_index("csv_code").to_dict(orient="index")
    labels = [shap_label(label_lookup[code]) for code in top_codes]

    rng = np.random.default_rng(RANDOM_SEED)
    fig, ax = plt.subplots(figsize=(11, 7))
    sc = None
    for y_pos, j in enumerate(top_indices):
        y = np.full(shap_values.shape[0], y_pos, dtype=float) + rng.normal(0, 0.08, shap_values.shape[0])
        values = feature_values[:, j]
        sc = ax.scatter(
            shap_values[:, j],
            y,
            c=values,
            cmap="coolwarm",
            s=12,
            alpha=0.65,
            edgecolors="none",
        )
    ax.axvline(0, color="black", linewidth=1, alpha=0.6)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("SHAP-style contribution to model logit")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    if sc is not None:
        cbar = fig.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label("imputed feature value")
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


plot_shap_bar(
    stump_shap_global_df,
    "mean_abs_shap_logit",
    "Gradient Boosting: Top 10 Additive SHAP-style Global Importance",
    "gradient_boosting_shap_style_top10_bar.png",
)

plot_shap_summary_dot(
    stump_shap_values,
    x_test_imputed,
    stump_shap_global_df,
    "Gradient Boosting: Additive SHAP-style Summary Plot",
    "gradient_boosting_shap_style_summary_dot.png",
)


## ALE Plots

Accumulated Local Effects (ALE) plots show how a feature changes the predicted probability locally, averaged over the test set. This complements SHAP: SHAP explains contributions per person; ALE shows the model response curve for one feature at a time. For gradient boosting, ALE is computed on the imputed raw-value model input scale.


In [ ]:
# ALE plots for gradient boosting.
# ALE uses local changes in predicted probability when one feature is moved within quantile bins.

ALE_TOP_N = 4
ALE_BINS = 10


def compute_ale_1d(x_matrix, predict_func, feature_idx, n_bins=10):
    values = np.asarray(x_matrix[:, feature_idx], dtype=float)
    finite = np.isfinite(values)
    values = values[finite]
    if len(values) == 0:
        return pd.DataFrame()

    unique_values = np.unique(values)
    if len(unique_values) < 2:
        return pd.DataFrame()

    if len(unique_values) <= n_bins:
        edges = unique_values
    else:
        edges = np.unique(np.quantile(values, np.linspace(0, 1, n_bins + 1)))
    if len(edges) < 2:
        return pd.DataFrame()

    rows = []
    local_effects = []
    counts = []
    centers = []
    for bin_idx in range(len(edges) - 1):
        lower = edges[bin_idx]
        upper = edges[bin_idx + 1]
        if bin_idx == len(edges) - 2:
            mask = finite & (x_matrix[:, feature_idx] >= lower) & (x_matrix[:, feature_idx] <= upper)
        else:
            mask = finite & (x_matrix[:, feature_idx] >= lower) & (x_matrix[:, feature_idx] < upper)
        n = int(mask.sum())
        if n == 0:
            local_effects.append(0.0)
            counts.append(0)
            centers.append((lower + upper) / 2)
            continue

        x_low = x_matrix[mask].copy()
        x_high = x_matrix[mask].copy()
        x_low[:, feature_idx] = lower
        x_high[:, feature_idx] = upper
        diff = predict_func(x_high) - predict_func(x_low)
        local_effects.append(float(np.mean(diff)))
        counts.append(n)
        centers.append(float((lower + upper) / 2))

    accumulated = np.cumsum(local_effects)
    weights = np.asarray(counts, dtype=float)
    if weights.sum() > 0:
        centered = accumulated - np.average(accumulated, weights=weights)
    else:
        centered = accumulated - np.mean(accumulated)

    for center, effect, local_effect, n in zip(centers, centered, local_effects, counts):
        rows.append({
            "feature_value_center": center,
            "ale_probability": float(effect),
            "local_probability_change": float(local_effect),
            "n": int(n),
        })
    return pd.DataFrame(rows)


def gb_predict_for_ale(x):
    return predict_gradient_boosting(x, gb_base_score, gb_stumps)

ale_features = perm_importance_df.head(ALE_TOP_N)["csv_code"].tolist()
ale_rows = []
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, feature in zip(axes, ale_features):
    j = feature_cols.index(feature)
    meta = feature_index.set_index("csv_code").loc[feature]
    ale_df = compute_ale_1d(x_test_imputed, gb_predict_for_ale, j, n_bins=ALE_BINS)
    ale_df.insert(0, "csv_code", feature)
    ale_df.insert(1, "ref_id", meta.get("ref_id"))
    ale_df.insert(2, "variable", meta.get("variable"))
    ale_df.insert(3, "survey_year", meta.get("survey_year"))
    ale_df.insert(4, "feature_group", meta.get("feature_group"))
    ale_df.insert(5, "question", meta.get("question"))
    ale_rows.append(ale_df)

    ax.plot(ale_df["feature_value_center"], ale_df["ale_probability"], marker="o")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(short_label(meta, max_width=38), fontsize=9)
    ax.set_xlabel("imputed feature value")
    ax.set_ylabel("ALE probability effect")

for ax in axes[len(ale_features):]:
    ax.axis("off")

fig.suptitle("Gradient Boosting: ALE Plots for Top Permutation Features", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "gradient_boosting_stumps_ale_top4.png", dpi=200, bbox_inches="tight")
plt.show()

ale_gb_df = pd.concat(ale_rows, ignore_index=True) if ale_rows else pd.DataFrame()
ale_gb_df.to_csv(OUT_DIR / "gradient_boosting_stumps_ale_top4.csv", index=False)
safe_display(ale_gb_df.head(12))
